In [ ]:
pip install pubchempy

In [3]:
import pandas as pd
import requests
import json

import pubchempy as pcp

In [6]:
import requests
import xml.etree.ElementTree as ET
import time

def consultar_chebi(chebi_id):
    # Endpoint REST para obter a entidade completa
    url = f"https://www.ebi.ac.uk/webservices/chebi/2.0/getCompleteEntity?chebiId={chebi_id}"
    try:
        response = requests.get(url)
        time.sleep(5.0)
        response.raise_for_status() # Verifica se houve erro na requisição

        # O ChEBI retorna XML. Vamos parsear:
        root = ET.fromstring(response.content)

        # Namespace do XML da EBI
        ns = {'chebi': 'http://www.ebi.ac.uk/webservices/chebi'}

        # Extração de dados básicos
        nome = root.find('.//chebi:chebiAsciiName', ns).text
        id_final = root.find('.//chebi:chebiId', ns).text

        print(f"--- Resultado para: {nome} ({id_final}) ---")

        # Listas para organizar os papéis e classes
        papeis_biologicos = []
        classes_quimicas = []
        sub_ontologias = []

        # Buscando relações na Ontologia
        for relation in root.findall('.//chebi:OntologyRelationships', ns):
            rel_type = relation.find('chebi:type', ns).text
            rel_name = relation.find('chebi:data', ns).text

            if rel_type == 'has role':
                papeis_biologicos.append(rel_name)
            elif rel_type == 'is a':
                classes_quimicas.append(rel_name)

        # Exibindo os resultados
        print(f"\n✅ Papéis Biológicos: {', '.join(papeis_biologicos) if papeis_biologicos else 'Nenhum encontrado'}")
        print(f"✅ Classes Químicas (Is a): {', '.join(classes_quimicas) if classes_quimicas else 'Nenhuma encontrada'}")

    except Exception as e:
        print(f"Erro ao consultar o ID {chebi_id}: {e}")

# Exemplo de uso: Cafeína (CHEBI:15377)
consultar_chebi("CHEBI:15377")

Erro ao consultar o ID CHEBI:15377: 500 Server Error: Internal Server Error for url: https://www.ebi.ac.uk/webservices/chebi/2.0/getCompleteEntity?chebiId=CHEBI:15377


In [7]:
import requests
import xml.etree.ElementTree as ET
import time

def consultar_chebi_robusto(chebi_id):
    # Removendo prefixo e garantindo que é só o número
    id_limpo = chebi_id.replace("CHEBI:", "")
    url = f"https://www.ebi.ac.uk/webservices/chebi/2.0/getCompleteEntity?chebiId={id_limpo}"

    # IMPORTANTE: Definir um Header para não ser bloqueado como bot
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
        'Accept': 'application/xml'
    }

    try:
        response = requests.get(url, headers=headers, timeout=15)

        if response.status_code == 500:
            print(f"❌ Erro 500 no ID {chebi_id}. O servidor da EBI está instável ou o ID é inválido.")
            return

        response.raise_for_status()

        # O resto do seu código de parse...
        root = ET.fromstring(response.content)
        # ... (continua igual ao seu código anterior)
        print(f"✅ Sucesso ao buscar {chebi_id}")

    except Exception as e:
        print(f"Erro: {e}")

consultar_chebi_robusto("15377")

❌ Erro 500 no ID 15377. O servidor da EBI está instável ou o ID é inválido.
